In [1]:

import numpy as np
import pandas as pd
import requests
from lib.odds import Odds

In [2]:
df = Odds().get_sports()

In [3]:
import pygwalker as pg

In [2]:
df = Odds().get_nba_champ_odds()

Number of events: 1

In [58]:
display(df.query("group == 'Basketball'"))

,key,group,title,description,active,has_outrights
5,basketball_euroleague,Basketball,Basketball Euroleague,Basketball Euroleague,True,False
6,basketball_nba,Basketball,NBA,US Basketball,True,False
7,basketball_nba_championship_winner,Basketball,NBA Championship Winner,Championship Winner 2024/2025,True,True
8,basketball_ncaab,Basketball,NCAAB,US College Basketball,True,False
9,basketball_ncaab_championship_winner,Basketball,NCAAB Championship Winner,US College Basketball Championship Winner,True,True


In [7]:
Odds.append_to_table(df, 'nba_champ_odds')

AttributeError: type object 'Odds' has no attribute 'append_to_table'

In [4]:
from datetime import datetime, timedelta

# Get current ISO time
current_iso_time = datetime.now().replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")

print("Current ISO Time: ", current_iso_time)

# Calculate 20 hours from now
future_time = datetime.now() + timedelta(hours=20)
future_iso_time = future_time.replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")

print("20 Hours From Now (ISO Time): ", future_iso_time)


Current ISO Time:  2024-11-13T12:42:26Z
20 Hours From Now (ISO Time):  2024-11-14T08:42:26Z


In [5]:


sport = 'basketball_nba'
odds_response = requests.get(
    f'https://api.the-odds-api.com/v4/sports/{sport}/odds',
    params={
        'api_key': '2b1fbcaa5cf9bbfca66c986129524664',
        'regions': 'us',
        'markets': 'h2h',
        'oddsFormat': 'american',
        'dateFormat': 'iso',
        'bookmakers': ['betmgm', 'fanduel', 'draftkings'],
        'commenceTimeFrom': current_iso_time,
        'commenceTimeTo': future_iso_time
    }
)

if odds_response.status_code != 200:
    print(f'Failed to get odds: status_code {odds_response.status_code}, response body {odds_response.text}')

else:
    odds_json = odds_response.json()
    print('Number of events:', len(odds_json))
    print(odds_json)

    # Check the usage quota
    print('Remaining requests', odds_response.headers['x-requests-remaining'])
    print('Used requests', odds_response.headers['x-requests-used'])

Number of events: 11
[{'id': 'da66702637cc0413553b92c792a05c50', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2024-11-14T00:10:00Z', 'home_team': 'Orlando Magic', 'away_team': 'Indiana Pacers', 'bookmakers': [{'key': 'betmgm', 'title': 'BetMGM', 'last_update': '2024-11-13T19:41:42Z', 'markets': [{'key': 'h2h', 'last_update': '2024-11-13T19:41:42Z', 'outcomes': [{'name': 'Indiana Pacers', 'price': -115}, {'name': 'Orlando Magic', 'price': -105}]}]}]}, {'id': '8e4ee6d0217de06a9defe24875b1b24e', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2024-11-14T00:40:00Z', 'home_team': 'Brooklyn Nets', 'away_team': 'Boston Celtics', 'bookmakers': [{'key': 'betmgm', 'title': 'BetMGM', 'last_update': '2024-11-13T19:41:42Z', 'markets': [{'key': 'h2h', 'last_update': '2024-11-13T19:41:42Z', 'outcomes': [{'name': 'Boston Celtics', 'price': -375}, {'name': 'Brooklyn Nets', 'price': 290}]}]}]}, {'id': 'ca4ada15a7c62966e17f7eecb36b2745', 'sport_key': 'bask

In [6]:
odds_meta = [
    ['id'],
    # ['has_outrights'],
    ['sport_key'],
    ['sport_title'],
    ['commence_time'],
    ['home_team'],
    ['away_team'],
    ['bookmakers', 'key'],
    ['bookmakers', 'title'],
    ['bookmakers', 'last_update'],
    ['bookmakers', 'markets', 'key'],
    ['bookmakers', 'markets', 'last_update'],
]

df = pd.json_normalize(odds_json, record_path=['bookmakers', 'markets', 'outcomes'], 
            meta=odds_meta, sep='_'
            )

In [7]:
pg.walk(df)

Box(children=(HTML(value='<div id="ifr-pyg-0" style="height: auto">\n    <head>\n        <meta http-equiv="Con…

In [8]:
display(df)

,name,price,id,sport_key,sport_title,commence_time,home_team,away_team,bookmakers_key,bookmakers_title,bookmakers_last_update,bookmakers_markets_key,bookmakers_markets_last_update
0,Indiana Pacers,-115,da66702637cc0413553b92c792a05c50,basketball_nba,NBA,2024-11-14T00:10:00Z,Orlando Magic,Indiana Pacers,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
1,Orlando Magic,-105,da66702637cc0413553b92c792a05c50,basketball_nba,NBA,2024-11-14T00:10:00Z,Orlando Magic,Indiana Pacers,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
2,Boston Celtics,-375,8e4ee6d0217de06a9defe24875b1b24e,basketball_nba,NBA,2024-11-14T00:40:00Z,Brooklyn Nets,Boston Celtics,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
3,Brooklyn Nets,290,8e4ee6d0217de06a9defe24875b1b24e,basketball_nba,NBA,2024-11-14T00:40:00Z,Brooklyn Nets,Boston Celtics,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
4,Chicago Bulls,230,ca4ada15a7c62966e17f7eecb36b2745,basketball_nba,NBA,2024-11-14T00:40:00Z,New York Knicks,Chicago Bulls,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
5,New York Knicks,-275,ca4ada15a7c62966e17f7eecb36b2745,basketball_nba,NBA,2024-11-14T00:40:00Z,New York Knicks,Chicago Bulls,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
6,Cleveland Cavaliers,-550,479ed48e101593d026f662ec12a3c984,basketball_nba,NBA,2024-11-14T00:40:00Z,Philadelphia 76ers,Cleveland Cavaliers,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
7,Philadelphia 76ers,400,479ed48e101593d026f662ec12a3c984,basketball_nba,NBA,2024-11-14T00:40:00Z,Philadelphia 76ers,Cleveland Cavaliers,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
8,New Orleans Pelicans,725,9c27acba13d14a8066e297096bce0349,basketball_nba,NBA,2024-11-14T00:40:00Z,Oklahoma City Thunder,New Orleans Pelicans,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
9,Oklahoma City Thunder,-1200,9c27acba13d14a8066e297096bce0349,basketball_nba,NBA,2024-11-14T00:40:00Z,Oklahoma City Thunder,New Orleans Pelicans,betmgm,BetMGM,2024-11-13T19:41:42Z,h2h,2024-11-13T19:41:42Z
